# Training recipe ablation

Measures what each component of the recipe is actually worth, rather than adopting the
standard bag of tricks because everyone else does.

## Quota

Kaggle's free tier is roughly 30 GPU hours per week and a 12 hour session cap. A full
30-epoch run is 2 to 4 hours, so six of them would eat most of a week and would not fit in
one session.

So the ladder below runs **15 epochs per configuration**, four configurations, at roughly
1 to 2 hours each. That is a comparison between configurations, not a leaderboard number.
The headline accuracy still comes from the full 30-epoch run in notebook 01 using whichever
configuration wins here.

Halving the epochs does bias the comparison, and in a predictable direction: mixup, cutmix,
and label smoothing are regularisers, and regularisation pays off later in training. A short
ablation therefore *understates* them. Worth remembering before concluding a component is
useless.

**Before running:** phone-verified account, Accelerator GPU T4 x2, Internet on.

In [ ]:
!git clone --depth 1 https://github.com/simonkundrik/plate-vision.git /kaggle/working/plate-vision
%pip install -q -e "/kaggle/working/plate-vision/model[train,export]"

import torch

print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit("No GPU. Set Accelerator to GPU T4 x2 before running.")

In [ ]:
%cd /kaggle/working/plate-vision/model
!python data/download_food101.py --out /kaggle/temp/food101

## The ladder

Cumulative rather than leave-one-out: each row adds one component to the row above. That
answers "is this worth adding", which is the decision actually being made, and costs four
runs instead of six.

In [ ]:
EPOCHS = 15
COMMON = (
    f"--data-root /kaggle/temp/food101/food-101 --backbone efficientnet_b0 "
    f"--epochs {EPOCHS} --batch-size 128 --lr 1e-3 --weight-decay 0.05 --workers 4 --amp"
)

LADDER = {
    "baseline": "",
    "+smoothing": "--label-smoothing 0.1",
    "+mixing": "--label-smoothing 0.1 --mixup-alpha 0.2 --cutmix-alpha 1.0",
    "+ema": "--label-smoothing 0.1 --mixup-alpha 0.2 --cutmix-alpha 1.0 --ema",
}

# The command is assembled in Python rather than interpolated straight into the `!` line.
# Static analysis cannot see inside shell magics, so building it here keeps the variables
# genuinely used and the cell lintable.
for name, extra in LADDER.items():
    out = f"/kaggle/working/runs/ablation/{name.replace('+', 'plus_')}"
    command = f"python scripts/train_classifier.py {COMMON} {extra} --out {out}"
    print(f"\n{'=' * 70}\n{name}\n{'=' * 70}", flush=True)
    !{command}

## Progressive resizing, measured on time rather than accuracy

This one is not an accuracy claim. The point is throughput: smaller images early mean more
epochs per GPU hour. Comparing its final accuracy against the others is only meaningful
alongside the wall-clock cost, so both get recorded.

In [ ]:
extra = LADDER["+ema"] + " --progressive-resize 128"
!python scripts/train_classifier.py {COMMON} {extra} --out /kaggle/working/runs/ablation/plus_resize

## The table

`val` is the live weights, `ema` is the averaged copy. Training accuracy is deliberately not
shown: under mixup the input is a blend of two images, so train top-1 is not the same
quantity as validation top-1 and putting them in one table invites a false comparison. The
`mixed` flag in `history.json` records which epochs were affected.

In [ ]:
import json
from pathlib import Path

import pandas as pd

rows = []
for run in sorted(Path("/kaggle/working/runs/ablation").iterdir()):
    history = json.loads((run / "history.json").read_text())
    val = [e for e in history if e["split"] == "val"]
    ema = [e for e in history if e["split"] == "ema"]
    train = [e for e in history if e["split"] == "train"]
    rows.append(
        {
            "run": run.name,
            "val top-1": round(max(e["top1"] for e in val), 2),
            "val top-5": round(max(e["top5"] for e in val), 2),
            "ema top-1": round(max((e["top1"] for e in ema), default=float("nan")), 2),
            "train min": round(sum(e["seconds"] for e in train) / 60, 1),
        }
    )

table = pd.DataFrame(rows)
table["delta"] = table["val top-1"].diff().round(2)
print(table.to_markdown(index=False))

## Reading the result honestly

A single 15-epoch run per configuration has real seed variance, plausibly a few tenths of a
percent. A delta smaller than that is not evidence of anything. Anything in that range should
be described as "no measurable effect at this budget", not as a win, and the honest fix is
repeated seeds rather than a confident narrative built on one run.

Copy this table into the top-level README with the epoch count stated next to it.